# Module 1 — Log Ingestion & Parsing (LogSentry)

**Project:** LogSentry — Incremental Vocabulary Adaptation for DeepLog
**Team:** B.Tech CSE Final Year Capstone
**Author:** Member 1

---

## What This Notebook Does

Reproduces the log-parsing stage of DeepLog (Du et al., CCS 2017) on the HDFS_v1 dataset:

1. Extract the raw HDFS log file from the shared dataset zip
2. Parse 11,175,629 log lines with the Drain3 algorithm into **34 unique log templates**
3. Group templates by HDFS block to build per-block **sequences of log keys**
4. Export sequences, labels, and vocabulary metadata for the LSTM training module (Member 2)

## Inputs
- `LogSentry_Capstone/01_Dataset/HDFS_v1.zip` (shared Drive)

## Outputs (written to `module1/data/processed/`)
- `deeplog_input.txt` — 575,061 sequences of log keys
- `deeplog_labels.txt` — block-level labels (Normal/Anomaly)
- `templates.csv` — log_key → template mapping
- `vocab_info.txt` — vocabulary metadata
- `parsed_logs.csv` — full line-level parse output (reference)

## Runtime
~15 minutes end-to-end on Colab free tier.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

paths = {
    'Private module1/':            '/content/drive/MyDrive/module1',
    'Private module1/code/':       '/content/drive/MyDrive/module1/code',
    'Shared LogSentry_Capstone/':  '/content/drive/MyDrive/LogSentry_Capstone',
    'Shared 02_Parsed_Data/':      '/content/drive/MyDrive/LogSentry_Capstone/02_Parsed_Data',
}

for name, p in paths.items():
    exists = os.path.exists(p)
    mark = '✅' if exists else '❌'
    print(f'{mark} {name:<30} {p}')

✅ Private module1/               /content/drive/MyDrive/module1
✅ Private module1/code/          /content/drive/MyDrive/module1/code
✅ Shared LogSentry_Capstone/     /content/drive/MyDrive/LogSentry_Capstone
✅ Shared 02_Parsed_Data/         /content/drive/MyDrive/LogSentry_Capstone/02_Parsed_Data


In [ ]:
!pip install drain3 tqdm pandas numpy --quiet

  Preparing metadata (setup.py) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pyiceberg 0.12.0 requires cachetools>=5.5, but you have cachetools 4.2.1 which is incompatible.


In [ ]:
import drain3, pandas, numpy, tqdm

print('drain3 installed')
print('pandas:', pandas.__version__)
print('numpy :', numpy.__version__)
print('tqdm  :', tqdm.__version__)

drain3 installed
pandas: 2.2.3
numpy : 2.1.3
tqdm  : 4.67.3


## Step 1 — Configuration

All paths and parser parameters are defined here so they can be changed
in one place.

In [ ]:
# ── Input paths ────────────────────────────────────────────────
ZIP_PATH = '/content/drive/MyDrive/LogSentry_Capstone/01_Dataset/HDFS_v1.zip'
ZIP_MEMBER_LOG      = 'HDFS.log'                      # path inside the zip
ZIP_MEMBER_LABELS   = 'preprocessed/anomaly_label.csv'

# ── Working files (Colab local disk, fast) ─────────────────────
LOCAL_LOG       = '/content/HDFS.log'
LOCAL_PARSED    = '/content/parsed_logs.csv'
LOCAL_TEMPLATES = '/content/templates.csv'

# ── Output destination (private Drive) ─────────────────────────
OUT_DIR = '/content/drive/MyDrive/module1/data/processed/'
OUT_RAW = '/content/drive/MyDrive/module1/data/raw/'

# ── Drain parser parameters ────────────────────────────────────
DRAIN_SIM_TH       = 0.4     # similarity threshold
DRAIN_DEPTH        = 4       # parse tree depth
DRAIN_MAX_CHILDREN = 100     # max children per node

import os
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(OUT_RAW, exist_ok=True)

print('Configuration ready.')
print(f'  Zip input:      {ZIP_PATH}')
print(f'  Local log:      {LOCAL_LOG}')
print(f'  Output folder:  {OUT_DIR}')
print(f'  Drain sim_th:   {DRAIN_SIM_TH}')

Configuration ready.
  Zip input:      /content/drive/MyDrive/LogSentry_Capstone/01_Dataset/HDFS_v1.zip
  Local log:      /content/HDFS.log
  Output folder:  /content/drive/MyDrive/module1/data/processed/
  Drain sim_th:   0.4


## Step 2 — Extract HDFS.log

Extract the 1.5 GB raw log from the shared zip to Colab's local disk.
We keep it on local disk (not Drive) because parsing does millions of
reads and Drive I/O is slower.

The step skips silently if the file is already extracted.

In [ ]:
import os, time, zipfile

if os.path.exists(LOCAL_LOG):
    size_gb = os.path.getsize(LOCAL_LOG) / (1024**3)
    print(f'✅ {LOCAL_LOG} already exists ({size_gb:.2f} GB) — skipping extraction.')
else:
    assert os.path.exists(ZIP_PATH), f'Zip not found: {ZIP_PATH}'
    print(f'Extracting {ZIP_MEMBER_LOG} from zip...')
    start = time.time()
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        z.extract(ZIP_MEMBER_LOG, path='/content/')
    elapsed = time.time() - start
    size_gb = os.path.getsize(LOCAL_LOG) / (1024**3)
    print(f'✅ Extracted in {elapsed:.1f}s  ({size_gb:.2f} GB)')

Extracting HDFS.log from zip...
✅ Extracted in 15.3s  (1.47 GB)


## Step 3 — Regex Masking

Drain clusters log lines by similarity. If we feed raw lines, block IDs
and IPs (which are unique per event) prevent lines from clustering,
producing hundreds of spurious templates.

We mask variable tokens first:
- Dates → `DATE`
- Block IDs → `BLK`
- IPs → `IP`
- Numbers → `NUM`

This lets structurally identical lines cluster into the same template.

In [ ]:
import re

MASK_PATTERNS = [
    (re.compile(r'^\d{6}\s+'),                              'DATE '),   # date at line start
    (re.compile(r'blk_-?\d+'),                              'BLK'),     # block IDs
    (re.compile(r'/(\d{1,3}\.){3}\d{1,3}(:\d+)?'),          'IP'),      # /IP:port
    (re.compile(r'\b\d{1,3}(\.\d{1,3}){3}\b'),              'IP'),      # bare IP
    (re.compile(r'(?<=[^A-Za-z0-9])\d+(?=[^A-Za-z0-9])'),   'NUM'),     # standalone numbers
]

BLOCK_RE = re.compile(r'(blk_-?\d+)')   # for extracting block_id from ORIGINAL line

def mask_line(line):
    for pattern, replacement in MASK_PATTERNS:
        line = pattern.sub(replacement, line)
    return line

# Sanity check
test = "081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010"
print('Before:', test)
print('After :', mask_line(test))

Before: 081109 203518 143 INFO dfs.DataNode$DataXceiver: Receiving block blk_-1608999687919862906 src: /10.250.19.102:54106 dest: /10.250.19.102:50010
After : DATE NUM NUM INFO dfs.DataNode$DataXceiver: Receiving block BLK src: IP dest: IP


## Step 4 — Full Parse

Parse all ~11 million lines with Drain3. For each line we record:
- `line_number` — position in the file
- `log_key` — template ID (from Drain's cluster_id — stable across runs)
- `block_id` — extracted from the original line, for grouping

Output written to Colab local disk first (fast). ~6 minutes.

In [ ]:
import time
from drain3 import TemplateMiner
from drain3.template_miner_config import TemplateMinerConfig
from tqdm import tqdm

config = TemplateMinerConfig()
config.drain_sim_th       = DRAIN_SIM_TH
config.drain_depth        = DRAIN_DEPTH
config.drain_max_children = DRAIN_MAX_CHILDREN
config.profiling_enabled  = False

miner = TemplateMiner(config=config)

# Count lines for the progress bar
with open(LOCAL_LOG, 'rb') as f:
    total_lines = sum(1 for _ in f)
print(f'Total lines: {total_lines:,}')

start = time.time()
parsed = 0
no_block = 0

with open(LOCAL_LOG, 'r', encoding='utf-8', errors='ignore') as f_in, \
     open(LOCAL_PARSED, 'w', encoding='utf-8') as f_out:

    f_out.write('line_number,log_key,block_id\n')

    for i, line in enumerate(tqdm(f_in, total=total_lines, desc='Parsing')):
        line = line.strip()
        if not line:
            continue

        # extract block_id from ORIGINAL line
        m = BLOCK_RE.search(line)
        block_id = m.group(1) if m else ''
        if not block_id:
            no_block += 1

        # parse the MASKED line
        masked = mask_line(line)
        result = miner.add_log_message(masked)
        log_key = result['cluster_id']

        f_out.write(f'{i},{log_key},{block_id}\n')
        parsed += 1

elapsed = time.time() - start
print(f'\n✅ Parsed {parsed:,} lines in {elapsed/60:.1f} minutes')
print(f'Unique templates: {len(miner.drain.clusters)}')
print(f'Lines without block_id: {no_block:,}')

Total lines: 11,175,629


Parsing: 100%|██████████| 11175629/11175629 [05:58<00:00, 31164.47it/s]


✅ Parsed 11,175,629 lines in 6.0 minutes
Unique templates: 34
Lines without block_id: 0


## Step 5 — Save Templates & Copy to Drive

Save the mapping `log_key → template_string`, then copy both
`parsed_logs.csv` and `templates.csv` from Colab local disk to
your private Drive folder.

In [ ]:
import os

with open(LOCAL_TEMPLATES, 'w', encoding='utf-8') as f:
    f.write('log_key,template\n')
    for cluster in miner.drain.clusters:
        t = cluster.get_template().replace('"', '""')
        f.write(f'{cluster.cluster_id},"{t}"\n')

size_kb = os.path.getsize(LOCAL_TEMPLATES) / 1024
print(f'✅ Templates saved: {LOCAL_TEMPLATES}  ({size_kb:.1f} KB)')
print(f'Total templates: {len(miner.drain.clusters)}')

✅ Templates saved: /content/templates.csv  (4.0 KB)
Total templates: 34


In [ ]:
import shutil, time, os

DRIVE_PARSED    = os.path.join(OUT_DIR, 'parsed_logs.csv')
DRIVE_TEMPLATES = os.path.join(OUT_DIR, 'templates.csv')

for src, dst in [(LOCAL_PARSED, DRIVE_PARSED), (LOCAL_TEMPLATES, DRIVE_TEMPLATES)]:
    print(f'Copying {os.path.basename(src)} ...')
    start = time.time()
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / (1024**2)
    print(f'  ✅ {size_mb:.1f} MB in {time.time()-start:.1f}s')

print()
print('Contents of', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    print(f'  {f}  ({os.path.getsize(p)/(1024**2):.2f} MB)')

Copying parsed_logs.csv ...
  ✅ 368.2 MB in 12.3s
Copying templates.csv ...
  ✅ 0.0 MB in 0.8s

Contents of /content/drive/MyDrive/module1/data/processed/
  README.md  (0.01 MB)
  deeplog_input.txt  (23.01 MB)
  deeplog_labels.txt  (17.23 MB)
  parsed_logs.csv  (368.18 MB)
  templates.csv  (0.00 MB)
  vocab_info.txt  (0.00 MB)


## Step 6 — Build Sequences for LSTM Training

DeepLog's LSTM consumes **sequences** of log keys, one sequence per HDFS
block, not individual lines. A block's sequence captures the full trace
of one file operation across datanodes.

This step:
1. Loads block-level labels from `anomaly_label.csv`
2. Groups parsed log keys by `block_id`
3. Writes `deeplog_input.txt` (sequences) and `deeplog_labels.txt` (labels)
4. Writes `vocab_info.txt` (metadata)

In [ ]:
import pandas as pd

# Extract labels from the zip (once) — small file, keep on raw/
LABELS_PATH = os.path.join(OUT_RAW, 'anomaly_label.csv')

if not os.path.exists(LABELS_PATH):
    import zipfile
    with zipfile.ZipFile(ZIP_PATH, 'r') as z:
        with z.open(ZIP_MEMBER_LABELS) as src, open(LABELS_PATH, 'wb') as dst:
            dst.write(src.read())
    print('✅ Extracted anomaly_label.csv from zip')

labels_df = pd.read_csv(LABELS_PATH)
print(f'Labels loaded: {len(labels_df):,} rows')
print(f'Columns: {list(labels_df.columns)}')
print(f'Label values: {labels_df["Label"].value_counts().to_dict()}')

Labels loaded: 575,061 rows
Columns: ['BlockId', 'Label']
Label values: {'Normal': 558223, 'Anomaly': 16838}


In [ ]:
from collections import defaultdict
from tqdm import tqdm
import numpy as np

with open(LOCAL_PARSED, 'rb') as f:
    total_rows = sum(1 for _ in f)
print(f'Total rows in parsed CSV: {total_rows:,}')

block_sequences = defaultdict(list)
line_count = 0
skipped = 0

with open(LOCAL_PARSED, 'r') as f:
    f.readline()  # header
    for line in tqdm(f, total=total_rows - 1, desc='Grouping by block'):
        parts = line.rstrip('\n').split(',')
        if len(parts) != 3:
            skipped += 1
            continue
        _, log_key_str, block_id = parts
        if not block_id:
            skipped += 1
            continue
        block_sequences[block_id].append(int(log_key_str))
        line_count += 1

lengths = [len(v) for v in block_sequences.values()]
print(f'\n✅ Unique blocks: {len(block_sequences):,}')
print(f'   Total log keys: {line_count:,}')
print(f'   Skipped: {skipped:,}')
print(f'   Sequence length — min: {min(lengths)}, max: {max(lengths)}, mean: {np.mean(lengths):.2f}, median: {int(np.median(lengths))}')

Total rows in parsed CSV: 11,175,630


Grouping by block: 100%|██████████| 11175629/11175629 [00:23<00:00, 478007.64it/s]



✅ Unique blocks: 575,061
   Total log keys: 11,175,629
   Skipped: 0
   Sequence length — min: 2, max: 298, mean: 19.43, median: 19


In [ ]:
import os
from tqdm import tqdm

label_dict = dict(zip(labels_df['BlockId'], labels_df['Label']))

LOCAL_SEQ   = '/content/deeplog_input.txt'
LOCAL_LABEL = '/content/deeplog_labels.txt'

written = 0
counts = {'Normal': 0, 'Anomaly': 0, 'Unknown': 0}

with open(LOCAL_SEQ, 'w') as f_seq, open(LOCAL_LABEL, 'w') as f_lab:
    f_lab.write('block_id,label\n')
    for block_id, keys in tqdm(block_sequences.items(), desc='Writing'):
        label = label_dict.get(block_id, 'Unknown')
        counts[label] += 1
        if label == 'Unknown':
            continue
        f_seq.write(' '.join(map(str, keys)) + '\n')
        f_lab.write(f'{block_id},{label}\n')
        written += 1

print(f'\n✅ Sequences written: {written:,}')
print(f'   Label distribution: {counts}')
print(f'   deeplog_input.txt : {os.path.getsize(LOCAL_SEQ)/(1024**2):.1f} MB')
print(f'   deeplog_labels.txt: {os.path.getsize(LOCAL_LABEL)/(1024**2):.1f} MB')

Writing: 100%|██████████| 575061/575061 [00:03<00:00, 152903.31it/s]


✅ Sequences written: 575,061
   Label distribution: {'Normal': 558223, 'Anomaly': 16838, 'Unknown': 0}
   deeplog_input.txt : 23.0 MB
   deeplog_labels.txt: 17.2 MB


In [ ]:
LOCAL_VOCAB = '/content/vocab_info.txt'

max_key = max(max(keys) for keys in block_sequences.values() if keys)
mean_len = sum(len(v) for v in block_sequences.values()) / len(block_sequences)
max_len  = max(len(v) for v in block_sequences.values())

with open(LOCAL_VOCAB, 'w') as f:
    f.write(f'vocab_size={max_key}\n')
    f.write(f'num_templates={len(miner.drain.clusters)}\n')
    f.write(f'num_sequences={written}\n')
    f.write(f'num_blocks_total={len(block_sequences)}\n')
    f.write(f'mean_sequence_length={mean_len:.2f}\n')
    f.write(f'max_sequence_length={max_len}\n')

print('vocab_info.txt:')
print('-' * 30)
with open(LOCAL_VOCAB) as f:
    print(f.read())

vocab_info.txt:
------------------------------
vocab_size=34
num_templates=34
num_sequences=575061
num_blocks_total=575061
mean_sequence_length=19.43
max_sequence_length=298



In [ ]:
import shutil, time

for fname in ['deeplog_input.txt', 'deeplog_labels.txt', 'vocab_info.txt']:
    src = f'/content/{fname}'
    dst = os.path.join(OUT_DIR, fname)
    print(f'Copying {fname}...')
    start = time.time()
    shutil.copy2(src, dst)
    size_mb = os.path.getsize(dst) / (1024**2)
    print(f'  ✅ {size_mb:.2f} MB in {time.time()-start:.1f}s')

print()
print('Final contents of', OUT_DIR)
for f in sorted(os.listdir(OUT_DIR)):
    p = os.path.join(OUT_DIR, f)
    print(f'  {f}  ({os.path.getsize(p)/(1024**2):.2f} MB)')

Copying deeplog_input.txt...
  ✅ 23.01 MB in 1.3s
Copying deeplog_labels.txt...
  ✅ 17.23 MB in 0.8s
Copying vocab_info.txt...
  ✅ 0.00 MB in 0.4s

Final contents of /content/drive/MyDrive/module1/data/processed/
  README.md  (0.01 MB)
  deeplog_input.txt  (23.01 MB)
  deeplog_labels.txt  (17.23 MB)
  parsed_logs.csv  (368.18 MB)
  templates.csv  (0.00 MB)
  vocab_info.txt  (0.00 MB)


## Step 7 — Final Verification

Confirms the outputs are internally consistent and ready for the LSTM
training module. If any check fails, do NOT hand off — investigate first.

In [ ]:
import os, numpy as np

print('=' * 60)
print('MODULE 1 — FINAL VERIFICATION')
print('=' * 60)

# ── 1. Files present ──
required = ['deeplog_input.txt', 'deeplog_labels.txt',
            'templates.csv', 'vocab_info.txt', 'parsed_logs.csv']
print('\n[1] Required files present:')
for f in required:
    p = os.path.join(OUT_DIR, f)
    mark = '✅' if os.path.exists(p) else '❌'
    size_mb = os.path.getsize(p)/(1024**2) if os.path.exists(p) else 0
    print(f'    {mark} {f:<25} {size_mb:>8.2f} MB')

# ── 2. Load & align ──
print('\n[2] Loading sequences and labels ...')
with open(os.path.join(OUT_DIR, 'deeplog_input.txt')) as f:
    sequences = [line.strip() for line in f]
with open(os.path.join(OUT_DIR, 'deeplog_labels.txt')) as f:
    f.readline()
    labels = [line.strip() for line in f]
print(f'    Sequences: {len(sequences):,}')
print(f'    Labels:    {len(labels):,}')
assert len(sequences) == len(labels), 'MISMATCH!'
print('    ✅ Counts match')

# ── 3. Validate log keys ──
parsed_seqs = [[int(k) for k in s.split()] for s in sequences]
all_keys = set(k for seq in parsed_seqs for k in seq)
print(f'\n[3] Log key range: {min(all_keys)} – {max(all_keys)}')
print(f'    Unique keys: {len(all_keys)}')
print(f'    Range valid (1–34): {min(all_keys) == 1 and max(all_keys) == 34}')
print(f'    No key 0 (padding safe): {0 not in all_keys}')

# ── 4. Sequence lengths ──
lengths = [len(s) for s in parsed_seqs]
print(f'\n[4] Sequence length: min={min(lengths)}  max={max(lengths)}  mean={np.mean(lengths):.2f}  median={int(np.median(lengths))}')

# ── 5. Labels ──
n_normal  = sum(1 for l in labels if l.endswith('Normal'))
n_anomaly = sum(1 for l in labels if l.endswith('Anomaly'))
total = n_normal + n_anomaly
print(f'\n[5] Labels: Normal={n_normal:,} ({100*n_normal/total:.2f}%)   Anomaly={n_anomaly:,} ({100*n_anomaly/total:.2f}%)')

print('\n' + '=' * 60)
print('✅ VERIFICATION PASSED — MODULE 1 READY FOR HANDOFF')
print('=' * 60)

MODULE 1 — FINAL VERIFICATION

[1] Required files present:
    ✅ deeplog_input.txt            23.01 MB
    ✅ deeplog_labels.txt           17.23 MB
    ✅ templates.csv                 0.00 MB
    ✅ vocab_info.txt                0.00 MB
    ✅ parsed_logs.csv             368.18 MB

[2] Loading sequences and labels ...
    Sequences: 575,061
    Labels:    575,061
    ✅ Counts match

[3] Log key range: 1 – 34
    Unique keys: 34
    Range valid (1–34): True
    No key 0 (padding safe): True

[4] Sequence length: min=2  max=298  mean=19.43  median=19

[5] Labels: Normal=558,223 (97.07%)   Anomaly=16,838 (2.93%)

✅ VERIFICATION PASSED — MODULE 1 READY FOR HANDOFF


## How to Use These Outputs (Member 2)

The files in `module1/data/processed/` are ready for LSTM training.

| File | Purpose |
|------|---------|
| `deeplog_input.txt` | One line per block — space-separated log keys |
| `deeplog_labels.txt` | `block_id,label` — same line order as input |
| `templates.csv` | `log_key → template string` (for interpretation) |
| `vocab_info.txt` | Vocabulary size = **34** |
| `parsed_logs.csv` | Raw line-level parse (reference only) |

**Critical points for training:**
- Log keys are **1–34**. `0` is reserved for padding.
- Embedding layer: `nn.Embedding(35, dim)` — one extra row for padding.
- Use `pad_sequence` with `padding_value=0`.
- Class imbalance: 2.93% anomalies → use class weights or focal loss.
- See `README.md` in `module1/data/processed/` for full PyTorch loader code
  and troubleshooting.

## Reproducibility

Run all cells top-to-bottom. Total runtime ~15 min on Colab free tier.
Outputs are deterministic given the same input zip and Drain parameters.